In [1]:
# After running this cell:
%pip uninstall mlflow -y
%pip install mlflow==2.9.2

# RESTART THE KERNEL HERE (Kernel → Restart from Jupyter menu)

Found existing installation: mlflow 2.9.2
Uninstalling mlflow-2.9.2:
  Successfully uninstalled mlflow-2.9.2
Note: you may need to restart the kernel to use updated packages.
  Using cached mlflow-2.9.2-py3-none-any.whl.metadata (13 kB)
Using cached mlflow-2.9.2-py3-none-any.whl (19.1 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os

In [2]:
%pwd

'/Users/dhruvkulshrestha/Desktop/MLOps_series/deep-learning-for-production-chest-cancer-classification-using-mlflow-dvc/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/dhruvkulshrestha/Desktop/MLOps_series/deep-learning-for-production-chest-cancer-classification-using-mlflow-dvc'

In [5]:
import dagshub
dagshub.init(repo_owner='Dhruvkulshrestha018', repo_name='deep-learning-for-production-chest-cancer-classification-using-mlflow-dvc', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)


mlflow.set_experiment("model evaluation")

Accessing as Dhruvkulshrestha018

Initialized MLflow to track repo 
"Dhruvkulshrestha018/deep-learning-for-production-chest-cancer-classification-using-mlflow-dvc"

Repository Dhruvkulshrestha018/deep-learning-for-production-chest-cancer-classification-using-mlflow-dvc 
initialized!

<Experiment: artifact_location='mlflow-artifacts:/afdad89a48444c80a840062a52e743b5', creation_time=1772045575593, experiment_id='1', last_update_time=1772045575593, lifecycle_stage='active', name='model evaluation', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [6]:
import tensorflow as tf
print(tf.__version__)

2.13.0


In [7]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [8]:
import mlflow

In [9]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [10]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [11]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/Chest-CT-Scan-data",
            mlflow_uri="https://dagshub.com/Dhruvkulshrestha018/deep-learning-for-production-chest-cancer-classification-using-mlflow-dvc.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [12]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse
%pip uninstall mlflow -y
%pip install mlflow==2.9.2

Found existing installation: mlflow 2.9.2
Uninstalling mlflow-2.9.2:
  Successfully uninstalled mlflow-2.9.2
Note: you may need to restart the kernel to use updated packages.
  Using cached mlflow-2.9.2-py3-none-any.whl.metadata (13 kB)
Using cached mlflow-2.9.2-py3-none-any.whl (19.1 MB)
Note: you may need to restart the kernel to use updated packages.


In [13]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
    
    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.30
        )
        
        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )
        
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
        
        # Check if directory exists
        if not Path(self.config.training_data).exists():
            raise FileNotFoundError(f"Training data directory not found: {self.config.training_data}")
            
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )
    
    def evaluation(self):
        # Check if model exists
        if not Path(self.config.path_of_model).exists():
            raise FileNotFoundError(f"Model not found: {self.config.path_of_model}")
            
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()
        self.log_into_mlflow()
    
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    
    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)
    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics({
                "loss": self.score[0],
                "accuracy": self.score[1]
            })
            
            if tracking_url_type_store != "file":
                mlflow.tensorflow.log_model(
                    self.model,
                    artifact_path="model",
                    registered_model_name="VGG16Model"
                )
            else:
                mlflow.tensorflow.log_model(
                    self.model,
                    artifact_path="model"
                )

In [14]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    
    print(f"Model path: {eval_config.path_of_model}")
    print(f"Data path: {eval_config.training_data}")
    
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    print("Evaluation completed successfully!")
    
except Exception as e:
    print(f"Error during evaluation: {str(e)}")
    import traceback
    traceback.print_exc()


[2026-02-26 03:47:23,071: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-02-26 03:47:23,073: INFO: common: yaml file: params.yaml loaded successfully]
[2026-02-26 03:47:23,074: INFO: common: created directory at: artifacts]
Model path: artifacts/training/model.h5
Data path: artifacts/data_ingestion/Chest-CT-Scan-data
Found 102 images belonging to 2 classes.
7/7 [==============================] - 11s 1s/step - loss: 0.0555 - accuracy: 0.9902
[2026-02-26 03:47:33,991: INFO: common: json file saved at: scores.json]


2026/02/26 03:47:35 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: /var/folders/17/_6fsxgvd79lfqd0b296vsbpm0000gn/T/tmptiqrxkrq/model/data/model/assets
[2026-02-26 03:47:36,490: INFO: builder_impl: Assets written to: /var/folders/17/_6fsxgvd79lfqd0b296vsbpm0000gn/T/tmptiqrxkrq/model/data/model/assets]


/opt/anaconda3/envs/chest-cancer/lib/python3.8/site-packages/_distutils_hack/__init__.py:31: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Successfully registered model 'VGG16Model'.
2026/02/26 03:48:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 1
Created version '1' of model 'VGG16Model'.


Evaluation completed successfully!


In [15]:
import tensorflow as tf
import mlflow

print("TF Version:", tf.__version__)
print("MLflow Version:", mlflow.__version__)

TF Version: 2.13.0
MLflow Version: 2.9.2
